# MLP for CIFAR10

This is very similar to the MNIST MLP. Notable changes will be highlighted below.

In [ ]:
!nvidia-smi

In [ ]:
import os
# NOTE!
# this is for a multi-gpu setup. we basically set which GPU is visible for the program.
# if you only have one GPU (most likely the case ;) ), make sure this is set to "0"!
# or remove this line completely!
# otherwise you may accidentally make your GPU "invisible" to the program!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt
from torch import nn

from idl.common import ClassifierTrainer, get_datasets_and_loaders
from idl.week1.analysis import plot_learning_curves
from idl.week3 import visualize_features

## Data

`torchvision` comes with the CIFAR datasets, so we just download it here.

It is generally a good idea to have a look at our data.
This is especially true if you include data augmentation -- you can make sure that your augmentations aren't too extreme.
Speaking of which, our `get_datasets_and_loaders` function accepts two optional arguments:
- `additional_transforms` can be a list of transforms that will be applied to training _and_ test data. This could be things like normalization transforms.
  Note that you do not have to use the `Compose` transform; this happens inside our helper function.
- `augment_transforms` will only be applied to the _training_ data. This should be used for data augmentation.
  You can simply create a list of your favorite transforms from `torchvision.transforms.v2` and pass them here!
  A few good/common ones include:
  - `RandomCrop` to a slightly smaller size, e.g. 30x30. The randomness leads to the objects essentially being randomly "moved" by a couple pixels.
  - `RandomHorizontalFlip` makes sense for CIFAR, but isn't always applicable. For example, flipping digits in MNIST doesn't make sense.
  - `RandomAffine` for rotation, translation (use this OR cropping, they effectively fulfill the same purpose), shearing and scaling ("zooming").
  - `ColorJitter` changes things like brighness, contrast, saturation or hue.

[Read the documentation](https://docs.pytorch.org/vision/main/transforms.html#) for details, as well as more transforms!
Note that for a small model, adding many transforms can result in a significant CPU bottleneck, slowing down your training a lot. Too bad!

In [ ]:
batch_size = 128
train_data, test_data, train_dataloader, test_dataloader = get_datasets_and_loaders("cifar10", batch_size=batch_size, num_workers=8)

We could also look a some dataset statistics again! We can see that the pixel vaule distribution is very different from MNIST; while there are still peaks near 0 and 1, we have plenty of values inbetween, with a sort-of Gaussian distribution. Also, the labels are actually balanced; there are 5000 images for each class.

In [ ]:
# first we gotta extract the data from the dataloader
images_np = []
labels_np = []

for image_batch, label_batch in train_dataloader:
    images_np.append(image_batch.numpy())
    labels_np.append(label_batch.numpy())

images_np = np.concatenate(images_np)
labels_np = np.concatenate(labels_np)

In [ ]:
# this code is as in the MNIST examples.
# so see those notebooks for some notes on the binning.
bins = np.arange(-0.5, 256.5, 1) / 255
plt.figure(figsize=(10, 4))
plt.hist(images_np.reshape(-1), bins=bins)
plt.title("Pixel distribution (linear)")
plt.xlabel("Pixel value")
plt.ylabel("Count")
plt.show()

# now the labels
bins = np.arange(-0.5, 10.5, 1)
plt.hist(labels_np, bins=bins)
plt.hlines(5000, -1, 10, colors="red", linestyles="dashed", label="Ideal balance")

plt.xticks(np.arange(10))
plt.xlim(-1, 10)
plt.title("Label distribution")
plt.xlabel("Label")
plt.ylabel("Count")
plt.legend()
plt.show()

In case you want to normalize the data (there is a `Normalize` transform), here are the mean and standard deviation:

In [ ]:
print(f"Total mean: {images_np.mean()} Total standard deviation: {np.std(images_np)}")

It's also common to use the per-channel mean/standard deviation instead, but this is more common with CNNs:

In [ ]:
print(f"Per-channel means: {images_np.mean(axis=(0, 2, 3))}")
print(f"Per-channel standard deviations: {np.std(images_np, axis=(0, 2, 3))}")

## Model

Here we build a simple MLP with three hidden layers with 1024 units each and GELU activation. This is by no means an ideal architecture, so you can tune it if you wish. It just serves as a starting point.
If you want to learn about the way we structure the model here, see `04_modules.ipynb`.

The `FCLayer` class is where you would include Batch Normalization and/or Dropout layers. Simply add them to `__init__` and be sure to remember to actually call them in `forward`.
- Normalization is most often placed after `Linear`, but before the activation function. Note that, as the mean is normalized to 0, the bias in the linear layer becomes redundant.
  You could pass `bias=False` to the `Linear` object to turn it off.
  - You could also try alternatives like `LayerNorm` or `GroupNorm`.
  - Be sure to [check the docs](https://docs.pytorch.org/docs/2.11/generated/torch.nn.BatchNorm1d.html).
- Dropout is most often placed after the activation function.

Note that both BN and dropout behave differently in training/inference, so it's extra important that we set our model to `train()` or `eval()` mode correctly.
This should generally be handled by our pre-existing training code.

In [ ]:
class FCLayer(nn.Module):
    def __init__(self,
                 n_inputs: int,
                 n_outputs: int):
        super().__init__()
        self.linear = nn.Linear(n_inputs, n_outputs)
        self.activation = nn.GELU()

    def forward(self,
                inputs: torch.Tensor) -> torch.Tensor:
        return self.activation(self.linear(inputs))


class MLP(nn.Module):
    def __init__(self,
                 n_inputs: int,
                 n_hidden: int,
                 n_outputs: int,
                 n_layers: int):
        super().__init__()
        self.root = nn.Flatten()
        
        self.body = nn.Sequential()
        for ind in range(n_layers):
            self.body.append(FCLayer(n_inputs if ind == 0 else n_hidden, n_hidden))
            
        self.head = nn.Linear(n_hidden, n_outputs)

    def forward(self,
                inputs: torch.Tensor) -> torch.Tensor:
        return self.head(self.body(self.root(inputs)))



device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")


n_hidden = 1024
model = MLP(n_inputs=32*32*3, n_hidden=n_hidden, n_outputs=10, n_layers=3).to(device).eval()
print(model)


def glorot_init(layer: nn.Module):
    if isinstance(layer, nn.Linear):
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)


with torch.no_grad():
    model.apply(glorot_init)

## Training

You should replace the optimizer. The easiest choice is `Adam` which you can probably just run with default parameters.
For regularization, you could switch to `AdamW` and tune the `weight_decay` parameter.
Strangely, adding some weight decay can sometimes even improve optimization!

This is also the place to add a learning rate scheduler.
Usually, we do _not_ want a constant learning rate;
rather, it should decay over time.
There are tons of options here, we recommend [reading the docs](https://docs.pytorch.org/docs/2.11/optim.html#how-to-adjust-learning-rate).
The easiest bang-for-buck option is probably `ReduceLROnPlateau`. You just have to tune the `factor` and `patience` parameters.
Don't forget to pass the scheduler to the `Trainer` object!  
Note that in our repository, we hardcoded schedulers to be called after _each training step_, with the exception of `ReduceLROnPlateau`, which is called after each _epoch_ on the validation loss.
This means that if you use a fixed scheduler (say, `CosineAnnealingLR`), you need to use the number of training steps, **not** epochs.
You can get this as the product of the number of epochs and the number of steps per epoch: `n_epochs * len(train_dataloader)`.

Another simple and basically free regularizer is _label smoothing_, where we replace the "hard" 0/1 labels by softer ones like 0.95, for example.
This is implemented as an argument to the `nn.CrossEntropyLoss`, and our `Trainer` accepts this argument, as well.

Finally, we implemented `EarlyStopping` in our repository. You can create an instance of that class here and also pass it to the `Trainer`.
As usual, we recommend that you have a look at the code to get an idea of how things like this can be implemented.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
trainer = ClassifierTrainer(model=model, optimizer=optimizer,
                            training_loader=train_dataloader, validation_loader=test_dataloader,
                            n_epochs=40, device=device,
                            classes=["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"])

In [ ]:
metrics = trainer.train_model()

In [ ]:
plot_learning_curves(metrics, keys=["cross_entropy", "accuracy"])

It's always a good idea at some classification results on the held-out data. We can see that the model does a good job an some examples, but fails badly on many others.

In [ ]:
trainer.plot_examples()

We can once again attempt to visualize the features learned by the first layer via plotting the weights in the shape of the input images. However, this becomes difficult to do "correctly" because we now have color images. With MNIST, we only had a single color channel, and so could plot the weights in different colors for positive/negative values. This no longer works with multiple input channels -- we have to plot the weights in RGB space. But this makes it impossible to really show positive/negative weights. The best we can do is somehow normalize the weights into the [0, 1] range. Then, bright values would indicate positive weights and dark values negative weights (for each color, respectively).

As such, don't worry too much about interpreting the plots below. Still, we will see that a properly trained and regularized model will have much more "distinct" features rather than the noisy mess we have right now.

In [ ]:
visualize_features(model.body[0].linear.weight.detach().cpu().numpy(), n_rows=32, data_shape=(3, 32, 32),
                   colormap="global", normalization="symmetric")